In [1]:
import pandas as pd
import re
import gc
import unicodedata
import random
from pathlib import Path

# ────────────────────────────────────────────────
# SETTINGS
# ────────────────────────────────────────────────

INPUT_FILE = r"H:\\data_cset312\\news_fake_real_only.csv"

REAL_OUTPUT  = r"H:\\data_cset312\\real_cleaned.csv"
FAKE_OUTPUT  = r"H:\\data_cset312\\fake_cleaned.csv"

TARGET_COUNT = 600_000

CHUNKSIZE    = 120_000
MIN_CHARS    = 150
MIN_WORDS    = 25

# ────────────────────────────────────────────────
# Cleaning function
# ────────────────────────────────────────────────

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return ""
    
    # Normalize unicode
    text = unicodedata.normalize("NFKD", text)
    
    # Remove various zero-width / control chars
    text = re.sub(r'[\u200B\u200C\u200D\uFEFF\u00AD\u2028\u2029]', '', text)
    
    # Collapse all whitespace to single space
    text = re.sub(r'\s+', ' ', text)
    
    # Strip
    text = text.strip()
    
    return text

# ────────────────────────────────────────────────
# Main logic
# ────────────────────────────────────────────────

real_collected  = []
fake_collected  = []

print("Reading input file in chunks and cleaning...")
print(f"Target per class: {TARGET_COUNT:,} rows")

total_read = 0

for chunk_num, chunk in enumerate(
    pd.read_csv(INPUT_FILE, chunksize=CHUNKSIZE, low_memory=True, encoding_errors='replace'),
    1
):
    total_read += len(chunk)
    print(f"Chunk {chunk_num:3d} | rows: {len(chunk):8,d}", end=" ... ")
    
    # Clean text
    chunk['text'] = chunk['text'].apply(clean_text)
    
    # Remove too short / empty after cleaning
    chunk = chunk[
        (chunk['text'].str.len() >= MIN_CHARS) &
        (chunk['text'].str.split().str.len() >= MIN_WORDS) &
        (chunk['text'].str.strip() != '')
    ]
    
    # Separate by label
    real_part  = chunk[chunk['label'] == 'real'].copy()
    fake_part  = chunk[chunk['label'] == 'fake'].copy()
    
    real_collected.extend(real_part.to_dict('records'))
    fake_collected.extend(fake_part.to_dict('records'))
    
    print(f"real: {len(real_part):6,d} | fake: {len(fake_part):6,d} | "
          f"cumul real: {len(real_collected):7,d} | cumul fake: {len(fake_collected):7,d}")
    
    del chunk, real_part, fake_part
    gc.collect()

# ────────────────────────────────────────────────
# Sampling exactly TARGET_COUNT rows per class
# ────────────────────────────────────────────────

def save_sampled(class_name, rows_list, output_path):
    if not rows_list:
        print(f"No {class_name} rows collected → skipping")
        return
    
    available = len(rows_list)
    to_take = min(TARGET_COUNT, available)
    
    print(f"\nSampling {class_name}: {to_take:,} from {available:,} available")
    
    random.shuffle(rows_list)
    selected = rows_list[:to_take]
    
    df = pd.DataFrame(selected)
    df = df[['text', 'label']]  # keep only needed columns
    
    # Final shuffle
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Saved → {output_path}  ({len(df):,} rows)")

# ────────────────────────────────────────────────
# Save both files
# ────────────────────────────────────────────────

save_sampled("real", real_collected, REAL_OUTPUT)
save_sampled("fake", fake_collected, FAKE_OUTPUT)

print("\n" + "═"*70)
print("Finished")
print(f"Total rows processed: {total_read:,}")
print(f"Created:")
print(f"  • {REAL_OUTPUT}")
print(f"  • {FAKE_OUTPUT}")

Reading input file in chunks and cleaning...
Target per class: 600,000 rows
Chunk   1 | rows:  120,000 ... real:  3,370 | fake: 113,084 | cumul real:   3,370 | cumul fake: 113,084
Chunk   2 | rows:  120,000 ... real: 11,676 | fake: 103,304 | cumul real:  15,046 | cumul fake: 216,388
Chunk   3 | rows:  120,000 ... real:  7,003 | fake: 109,133 | cumul real:  22,049 | cumul fake: 325,521
Chunk   4 | rows:  120,000 ... real:  4,261 | fake: 109,877 | cumul real:  26,310 | cumul fake: 435,398
Chunk   5 | rows:  120,000 ... real:    571 | fake: 116,537 | cumul real:  26,881 | cumul fake: 551,935
Chunk   6 | rows:  120,000 ... real:  1,444 | fake: 112,209 | cumul real:  28,325 | cumul fake: 664,144
Chunk   7 | rows:  120,000 ... real:  1,724 | fake: 110,104 | cumul real:  30,049 | cumul fake: 774,248
Chunk   8 | rows:  120,000 ... real: 33,803 | fake: 80,521 | cumul real:  63,852 | cumul fake: 854,769
Chunk   9 | rows:  120,000 ... real: 118,041 | fake:      0 | cumul real: 181,893 | cumul fak

In [2]:
import pandas as pd
import os
from pathlib import Path

# Define the two files (change paths if needed)
files = {
    "Real":  r"H:\\data_cset312\\real_cleaned.csv",
    "Fake": r"H:\\data_cset312\\fake_cleaned.csv"
}

def check_file_details(label, filepath):
    print("\n" + "=" * 70)
    print(f"FILE: {label}  →  {filepath}")
    print("-" * 70)

    # 1. File existence & size
    if not os.path.exists(filepath):
        print("→ File does NOT exist!")
        return

    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"File size: {size_mb:.1f} MB")

    # 2. Load data (small preview + stats)
    try:
        df = pd.read_csv(filepath, low_memory=True, encoding_errors='replace')

        # Number of rows
        row_count = len(df)
        print(f"Number of rows: {row_count:,}")

        # Columns
        print(f"Columns: {list(df.columns)}")

        # Label distribution (should be almost 100% one class)
        if 'label' in df.columns:
            print("\nLabel distribution:")
            print(df['label'].value_counts(dropna=False))
            print("\nPercentages:")
            print(df['label'].value_counts(normalize=True).mul(100).round(2).astype(str) + " %")

        # Text length statistics
        if 'text' in df.columns:
            df['text_length_chars'] = df['text'].astype(str).str.len()
            df['text_length_words'] = df['text'].astype(str).str.split().str.len()

            print("\nText length statistics:")
            print(df[['text_length_chars', 'text_length_words']].describe().round(0).astype(int))

            # Longest & shortest previews
            print("\nLongest text preview (first 200 chars):")
            longest = df.loc[df['text_length_chars'].idxmax(), 'text']
            print(f"{longest[:200]}...")

            print("\nShortest text preview (first 200 chars):")
            shortest = df.loc[df['text_length_chars'].idxmin(), 'text']
            print(f"{shortest[:200]}...")

        # Show first 5 rows
        print("\nFirst 5 rows:")
        print(df.head(5)[['text', 'label']].to_string(index=False))

    except Exception as e:
        print(f"Error reading file: {e}")

# Run check for both files
for name, path in files.items():
    check_file_details(name, path)

print("\n" + "=" * 70)
print("Check complete.")


FILE: Real  →  H:\\data_cset312\\real_cleaned.csv
----------------------------------------------------------------------
File size: 1772.1 MB
Number of rows: 600,000
Columns: ['text', 'label']

Label distribution:
label
real    600000
Name: count, dtype: int64

Percentages:
label
real    100.0 %
Name: proportion, dtype: str

Text length statistics:
       text_length_chars  text_length_words
count             600000             600000
mean                3062                509
std                 3337                554
min                  150                 25
25%                  965                158
50%                 2150                356
75%                 4327                724
max               187662              32379

Longest text preview (first 200 chars):
How Could Obama Voters Go for Trump? 😂😂 A photo posted by Daquan Gesese (@daquan) on Nov 13, 2016 at 5:59pm PST I seriously can’t think of more polar-opposite people and public figures than Donald J. ...

Shorte